In [ ]:
pip install python-vlc

In [4]:
# 通过单词，查询对应的美式英语音标和英式英语音标。
# 注意更换 json file 的文件路径。
import re
import json

def extract_pronunciation(word):
    with open('/Users/chengyong/Downloads/cam_dict.refined.json', 'r') as f:
        for line in f:
            try:
                entry = json.loads(line)
                if entry['word'] == word:
                    pronunciations = entry['pos_items'][0]['pronunciations']
                    for pronunciation in pronunciations:
                        region = pronunciation['region']
                        ipa = pronunciation['pronunciation']
                        print(f"Region: {region}, Pronunciation: {ipa}")
                    return
            except json.JSONDecodeError:
                continue

    print(f"No pronunciation found for '{word}'")

# 使用示例
extract_pronunciation('terrible')
extract_pronunciation('stable')
extract_pronunciation('able')

Region: uk, Pronunciation: ˈter.ə.bəl
Region: us, Pronunciation: ˈter.ə.bəl
Region: uk, Pronunciation: ˈsteɪ.bəl
Region: us, Pronunciation: ˈsteɪ.bəl
Region: uk, Pronunciation: ˈeɪ.bəl
Region: us, Pronunciation: ˈeɪ.bəl


In [ ]:
extract_pronunciation('thirtieth')

In [13]:
# 基于上面代码修改，只导出us pronunciation
import json

def extract_us_pronunciation(word):
    with open('/Users/chengyong/Downloads/cam_dict.refined.json', 'r') as f:
        for line in f:
            try:
                # Load each line as JSON
                entry = json.loads(line)

                # Check if the word matches
                if entry['word'] == word:
                    # Get the pronunciations list from the first pos_item
                    pronunciations = entry['pos_items'][0]['pronunciations']

                    # Iterate through the pronunciations
                    for pronunciation in pronunciations:
                        region = pronunciation.get('region')
                        ipa = pronunciation.get('pronunciation')

                        # Only print US pronunciation
                        if region == 'us':
                            print(f"US Pronunciation of '{word}': {ipa}")
                            return

            except json.JSONDecodeError:
                # Continue to next line if there is a JSON parsing error
                continue

    # If no pronunciation is found
    print(f"No US pronunciation found for '{word}'")

# Usage example
extract_us_pronunciation('terrible')
extract_us_pronunciation('stable')
extract_us_pronunciation('able')


US Pronunciation of 'terrible': ˈter.ə.bəl
US Pronunciation of 'stable': ˈsteɪ.bəl
US Pronunciation of 'able': ˈeɪ.bəl


In [18]:
# 导入excel，批量输出 us发音，并写回csv文件

import json
import pandas as pd

# Function to extract US pronunciation for a given word
def extract_us_pronunciation(word):
    with open('/Users/chengyong/Downloads/cam_dict.refined.json', 'r') as f:
        for line in f:
            try:
                # Load each line as JSON
                entry = json.loads(line)

                # Check if the word matches
                if entry.get('word') == word:
                    # Get the pronunciations list from the first pos_item
                    pos_items = entry.get('pos_items', [])

                    # Check if pos_items exists and is not empty
                    if pos_items:
                        pronunciations = pos_items[0].get('pronunciations', [])

                        # Iterate through the pronunciations
                        for pronunciation in pronunciations:
                            region = pronunciation.get('region')
                            ipa = pronunciation.get('pronunciation')

                            # Only return US pronunciation
                            if region == 'us':
                                return ipa
            except json.JSONDecodeError:
                # Continue to next line if there is a JSON parsing error
                continue

    # If no pronunciation is found, return None
    return None

# Load the CSV file into a DataFrame
csv_file_path = '/Users/chengyong/Desktop/hard_pronunciation_word_list.csv'
df = pd.read_csv(csv_file_path)

# Ensure the CSV has a column named 'word'
if 'word' not in df.columns:
    raise ValueError("CSV file must contain a 'word' column.")

# Add a new column to store US pronunciations if it doesn't already exist
if 'us_pronunciation' not in df.columns:
    df['us_pronunciation'] = None

# Iterate over each row in the DataFrame, extract pronunciation and update the DataFrame
for index, row in df.iterrows():
    word = row['word']

    # Skip rows that already have pronunciations
    if pd.notna(row['us_pronunciation']):
        continue

    us_pronunciation = extract_us_pronunciation(word)
    if us_pronunciation:
        print(f"Extracted US pronunciation for '{word}': {us_pronunciation}")  # Debugging output
    else:
        print(f"No US pronunciation found for '{word}'")  # Debugging output

    # Update the DataFrame
    df.at[index, 'us_pronunciation'] = us_pronunciation

# Save the updated DataFrame back to the CSV file
df.to_csv(csv_file_path, index=False)

print("Pronunciations have been successfully extracted and saved to the CSV file.")



Extracted US pronunciation for 'access': ˈæk.ses


/var/folders/yp/qyzd_0dx27vblg19cv45z67h0000gn/T/ipykernel_57721/856562041.py:65: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ˈæk.ses' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, 'us_pronunciation'] = us_pronunciation


No US pronunciation found for 'Adobe'
Extracted US pronunciation for 'twelve': twelv
No US pronunciation found for '25th'
Extracted US pronunciation for 'absent': ˈæb.sənt
Extracted US pronunciation for 'achieve': əˈtʃiːv
Extracted US pronunciation for 'actually': ˈæk.tʃu.ə.li
Extracted US pronunciation for 'added': ˈæd.ɪd
Extracted US pronunciation for 'addict': ˈæd.ɪkt
No US pronunciation found for 'admin'
Extracted US pronunciation for 'admit': ədˈmɪt
Extracted US pronunciation for 'adversarial': ˌæd.vɚˈser.i.əl
Extracted US pronunciation for 'again': əˈɡen
Extracted US pronunciation for 'agile': ˈædʒ.əl
No US pronunciation found for 'AJAX'
Extracted US pronunciation for 'alias': ˈeɪ.li.əs
Extracted US pronunciation for 'align': əˈlaɪn
Extracted US pronunciation for 'also': ˈɑːl.soʊ
Extracted US pronunciation for 'always': ˈɑːl.weɪz
Extracted US pronunciation for 'amazon': ˈæm.ə.zɑːn
Extracted US pronunciation for 'analogy': əˈnæl.ə.dʒi
Extracted US pronunciation for 'anchor': ˈæŋ.k

In [19]:
# 统计csv文件中单词发音的元音、辅音出现的频率

import pandas as pd
import re
from collections import Counter

# 元音和辅音列表
vowels = [
    "ə", "ɚ", "ɝː", "ʌ", "ɑː", "ɑːr", "e", "ɛ", "æ", "er", "ɪ", "iː", "i", "ɪr",
    "ɔː", "ɔːr", "ʊ", "u", "uː", "ʊr", "aɪ", "aɪr", "eɪ", "ɔɪ", "aʊ", "aʊr", "oʊ"
]
consonants = [
    "p", "b", "t", "t̬", "d", "k", "g", "f", "v", "s", "z", "θ", "ð", "ʃ", "ʒ",
    "tʃ", "dʒ", "tr", "dr", "ts", "dz", "m", "n", "ŋ", "l", "r", "w", "j", "h"
]

# 加载 CSV 文件
csv_file_path = '/Users/chengyong/Desktop/hard_pronunciation_word_list.csv'
df = pd.read_csv(csv_file_path)

# 确保有 'us_pronunciation' 列
if 'us_pronunciation' not in df.columns:
    raise ValueError("CSV file must contain a 'us_pronunciation' column.")

# 初始化计数器
vowel_counter = Counter()
consonant_counter = Counter()

# 遍历每个发音并统计元音和辅音的频率
for pronunciation in df['us_pronunciation']:
    if pd.isna(pronunciation):  # 跳过没有发音的行
        continue

    # 统计元音频率
    for vowel in vowels:
        occurrences = len(re.findall(re.escape(vowel), pronunciation))
        if occurrences > 0:
            vowel_counter[vowel] += occurrences

    # 统计辅音频率
    for consonant in consonants:
        occurrences = len(re.findall(re.escape(consonant), pronunciation))
        if occurrences > 0:
            consonant_counter[consonant] += occurrences

# 打印统计结果
print("元音频率统计:")
for vowel, count in vowel_counter.most_common():
    print(f"{vowel}: {count}")

print("\n辅音频率统计:")
for consonant, count in consonant_counter.most_common():
    print(f"{consonant}: {count}")


元音频率统计:
ɪ: 246
ə: 216
e: 146
i: 101
æ: 72
ɑː: 63
eɪ: 54
iː: 49
ʊ: 47
ɚ: 45
ʌ: 35
aɪ: 30
oʊ: 26
u: 25
uː: 22
ɝː: 19
aʊ: 14
ɔː: 12
ɔːr: 12
er: 8
ɑːr: 6
ɪr: 5
ʊr: 4
ɔɪ: 3
aʊr: 1

辅音频率统计:
l: 214
t: 182
n: 172
s: 141
d: 128
r: 117
k: 109
p: 68
m: 64
f: 58
b: 51
v: 41
ŋ: 37
ʃ: 33
w: 30
ʒ: 28
t̬: 28
z: 25
dʒ: 24
h: 22
j: 18
tʃ: 17
tr: 14
θ: 13
ð: 11
dr: 5


In [12]:
# 查询 New Ofxord Dictionary.json 中的单词
import json
import re

def get_phonetic(word, json_data):
    """
    从 JSON 数据中查找指定单词的音标信息。
    
    参数:
    word (str): 需要查找的单词
    json_data (str): 包含单词及其音标信息的 JSON 数据
    
    返回值:
    str: 找到的音标信息,如果没找到则返回 'Not found'
    """
    # 使用正则表达式匹配单词及其音标
    pattern = fr'"({word})":"([^"]+)"'
    match = re.search(pattern, json_data)
    
    if match:
        # 从匹配结果中提取音标信息
        phonetic = match.group(2).split('|')[1].strip()
        return phonetic
    else:
        return 'Not found'

# 示例用法
with open('/Users/chengyong/Documents/07_图书馆/Mastering the American Accent/New Oxford American Dictionary.json', 'r') as f:
    json_data = f.read()

print(get_phonetic('mountain', json_data))  # Output: ˌdəbəl ˌō ˈsevənˌdəbəl ˌoʊ ˈsɛvən



ˈmount(ə)nˈmaʊnt(ə)n


In [ ]:
# 通过音标，找匹配的单词，只去找美国发音音标对应的英文单词。
import json
import re

def query_ipa_words(ipa_pattern):
    matches = []
    with open('/Users/chengyong/Documents/01_Coding/Github-desktop-sync/macbook_computer_code/cam_dict.refined.json', 'r') as f:
        for line in f:
            try:
                data = json.loads(line)
                word = data['word']
                pronunciations = [p for p in data['pos_items'][0]['pronunciations'] if p['region'] == 'us']
                for pronunciation in pronunciations:
                    ipa = pronunciation['pronunciation']
                    if re.search(ipa_pattern, ipa):
                        matches.append((word, ipa))
            except (KeyError, IndexError):
                # 跳过无效或缺失数据的行
                continue

    return matches

# 查询包含 'eɪ.ʃən' 的单词
# 这里可以充分利用正则表达式，做一些排除，譬如我只想查询包含单独的ʒ的单词，而不是 dʒ,于是用正则表达式 r'(?<!d)ʒ'
# ipa_pattern = r'(?<!d)ʒ'

# 寻找类似 tower json 中注音为/taʊ.ɚ/, flower /flaʊ.ɚ/
# ipa_pattern = r'ʊ.ɚ'

ipa_pattern = r'edʒ'
results = query_ipa_words(ipa_pattern)
for word, ipa in results:
    print(f"Word: {word}, IPA: {ipa}")

In [ ]:
#查询弹舌音 t
# json 文件中，water 的美式发音为：wɑː.t̬ɚ

ipa_pattern = r't̬'
results = query_ipa_words(ipa_pattern)
for word, ipa in results:
    print(f"Word: {word}, IPA: {ipa}")